# nano-shatta — train a Shatta Wale-style lyrics model

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** with **LoRA via Unsloth**
(2x faster, ~70% less VRAM) on 938 dancehall songs
(2MB corpus, already in this repo as `shatta_clean_v3.txt`).

- **Cost:** free (Colab T4 GPU)
- **Time:** ~10–20 min for 3 epochs (Unsloth speedup)
- **Output:** `shatta-lora.zip` (~50MB adapter) you download at the end

**What you get:** a model that writes Shatta-Wale-style bars.

Run the cells **top to bottom**. Each section says what it does and what to expect.

## 1 · Setup

Clones this repo into the Colab VM and installs dependencies.
(If you forked this repo, edit `REPO_URL` below to point at your fork.)

In [ ]:
REPO_URL = 'https://github.com/Ayehnor99/nano-shatta.git'  # edit only if you forked this repo
import os, subprocess
# Idempotent: always land at /content/nano-shatta, however many times you
# re-run this cell (no more nano-shatta/nano-shatta/nano-shatta nesting).
os.chdir("/content")
if not os.path.isdir("/content/nano-shatta/.git"):
    subprocess.run(["git", "clone", REPO_URL, "/content/nano-shatta"], check=True)
os.chdir("/content/nano-shatta")
subprocess.run(["git", "pull", "-q"])
!pip install -q unsloth
!pip install -q -r requirements.txt
print("setup done — repo at", os.getcwd())

## 2 · Keep-alive (prevents idle disconnect)

Colab kills free sessions after ~90 min idle. This cell starts a keep-alive
in a **background thread** and finishes immediately — run it and move on.
(During active training the GPU keeps the session alive; keep-alive only
matters if you step away mid-run.)

In [ ]:
import threading, time
import IPython.display as ipd

def _keep_alive():
    while True:
        ipd.clear_output(wait=True)
        print("keep-alive", time.strftime("%H:%M:%S"))
        time.sleep(60)

threading.Thread(target=_keep_alive, daemon=True).start()
print("keep-alive running in background — this cell finishes immediately")

## 3 · Check the GPU

You must see a GPU name (T4 / L4 / A100). If it says `NO GPU`:
**Runtime → Change runtime type → T4 GPU**, then re-run this cell.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
else:
    print("NO GPU — go to Runtime -> Change runtime type -> T4 GPU")

## 4 · Train (Unsloth LoRA, 4-bit) — with LIVE monitoring

We fine-tune **Qwen2.5-1.5B-Instruct** (pre-quantized 4-bit by Unsloth for
fast download) using Unsloth's optimized kernels — ~2x faster and ~70% less
VRAM than the vanilla HF path.

You'll see a one-line status every 5 steps with loss, lr, tok/s, elapsed,
ETA, and a live **sparkline** of the loss curve:

```
step   100/170 | loss 2.8431 | lr 1.90e-04 | 8,192 tok/s | elapsed 03:12 | ETA 02:41 | ▁▂▂▃▄▄▅▅▆▇
```

Every `--sample-every` steps (set to 50 below) it also prints a **val loss**
and generates a **sample bar** from the mid-training model — you can
literally watch it learn to write Shatta-style lyrics. Samples are also
appended to `shatta-lora/samples.log`.

- 3 epochs over ~900 blocks of 512 tokens (effective batch 16)
- 5% of blocks held out for val loss
- Loss should drop from ~4.0 toward ~2.0 or lower
- Checkpoints save every 100 steps to `shatta-lora/checkpoints/`

If the session dies mid-training, re-run this cell with `--resume` appended.

In [ ]:
!python train_unsloth.py \
    --model unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit \
    --data shatta_clean_v3.txt \
    --out shatta-lora \
    --epochs 3 \
    --batch-size 2 \
    --grad-accum 8 \
    --block-size 512 \
    --lr 2e-4 \
    --lora-r 16 \
    --lora-alpha 32 \
    --save-steps 100 \
    --val-frac 0.05 \
    --sample-every 50 \
    --sample-prompt "[Verse 1]" \
    --sample-tokens 48

## 5 · Generate a test bar

Loads the freshly trained adapter and writes a verse. The base model is
read **from the adapter config** — no manual base-model flag, so the
train/generate base mismatch bug can't happen.
Edit the `--prompt` to try your own opening line.

In [ ]:
!python generate.py --adapter shatta-lora \
    --prompt "[Verse 1] Mi a di realest" \
    --max-new-tokens 100 --seed 7

## 6 · Generate a WHOLE song

Structure-aware generation: the model writes each section in turn
(Intro → Verse → Chorus → Verse → Chorus → Bridge → Outro), carrying the
song so far into every prompt so it stays on theme. `--repetition-penalty`
stops it looping the same line.

Edit `--title` and `--structure` to taste. Add `--seed 3` to reproduce a run.

In [ ]:
!python generate.py --adapter shatta-lora --song \
    --title "Street Crown" \
    --structure "Intro,Verse 1,Chorus,Verse 2,Chorus,Bridge,Outro" \
    --section-tokens 90 --temperature 0.9 --repetition-penalty 1.15 \
    --out song.txt
print(open("song.txt").read())

## 7 · Download the adapter

Zips `shatta-lora/` and downloads it. It contains `adapter_config.json`,
`adapter_model.safetensors`, and the tokenizer files — the whole trained model,
~50MB. Load it on top of the base Qwen2.5-1.5B-Instruct anywhere (see README).
The adapter config records the exact base, so generation always uses the
right one.

In [ ]:
!zip -q -r shatta-lora.zip shatta-lora
from google.colab import files
files.download("shatta-lora.zip")

## 8 · (Optional) Back up to Google Drive

If the session dies mid-training, Colab's VM is wiped. Mount Drive and copy
the output so a `--resume` can pick up where you left off.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/nano-shatta-backup
!cp -r shatta-lora /content/drive/MyDrive/nano-shatta-backup/
print("backed up to Drive")